In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make src/ importable from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import src.data_loader
import src.features
import src.train


from src.data_loader import fetch_dam_prices, load_prices, save_prices
from src.features import (
    build_supervised_dataset,
    make_calendar_features,
    make_forecast_time_features,
    make_target_relative_lags,
    hours_to_target,
)
from src.train import train_per_horizon_models, predict_next_day

from src.processing import ensure_hourly, ensure_hourly_df, process_raw_csv,_detect_subhourly

print("Imports OK.")

Imports OK.


In [32]:
path = Path(r"C:\dev\greek-dam-forecasting\data\raw\backfill_tmp\dam_prices_historical.csv")
output_path = Path(r"C:\dev\greek-dam-forecasting\data\processed\dam_prices_historical_hourly.csv")
df_raw = pd.read_csv(path)
df_raw

,Unnamed: 0,price_eur_mwh
0,2022-12-31 22:00:00+00:00,210.05
1,2022-12-31 23:00:00+00:00,230.90
2,2023-01-01 00:00:00+00:00,268.19
3,2023-01-01 01:00:00+00:00,229.58
4,2023-01-01 02:00:00+00:00,235.98
...,...,...
45016,2026-05-06 20:00:00+00:00,155.83
45017,2026-05-06 20:15:00+00:00,148.44
45018,2026-05-06 20:30:00+00:00,171.42
45019,2026-05-06 20:45:00+00:00,171.14


In [33]:
df = pd.read_csv(path, index_col=0)
df

,price_eur_mwh
2022-12-31 22:00:00+00:00,210.05
2022-12-31 23:00:00+00:00,230.90
2023-01-01 00:00:00+00:00,268.19
2023-01-01 01:00:00+00:00,229.58
2023-01-01 02:00:00+00:00,235.98
...,...
2026-05-06 20:00:00+00:00,155.83
2026-05-06 20:15:00+00:00,148.44
2026-05-06 20:30:00+00:00,171.42
2026-05-06 20:45:00+00:00,171.14


In [19]:
res = process_raw_csv(input_path=path, output_path=output_path)
df_processed = res["df"]

In [20]:
df_processed

,price_eur_mwh
2022-12-31 22:00:00+00:00,210.0500
2022-12-31 23:00:00+00:00,230.9000
2023-01-01 00:00:00+00:00,268.1900
2023-01-01 01:00:00+00:00,229.5800
2023-01-01 02:00:00+00:00,235.9800
...,...
2026-05-07 17:00:00+00:00,160.5125
2026-05-07 18:00:00+00:00,130.0750
2026-05-07 19:00:00+00:00,122.9725
2026-05-07 20:00:00+00:00,119.3350


In [21]:
path = Path(r"C:\dev\greek-dam-forecasting\data\raw\backfill_tmp\load_forecast_historical.csv")
output_path = Path(r"C:\dev\greek-dam-forecasting\data\processed\forecast_historical_hourly.csv")

In [22]:
df_raw = pd.read_csv(path)
df_raw

,Unnamed: 0,load_forecast_mw
0,2022-12-31 22:00:00+00:00,4269.0
1,2022-12-31 23:00:00+00:00,4008.0
2,2023-01-01 00:00:00+00:00,3888.0
3,2023-01-01 01:00:00+00:00,3760.0
4,2023-01-01 02:00:00+00:00,3703.0
...,...,...
42080,2026-05-08 19:45:00+00:00,5020.0
42081,2026-05-08 20:00:00+00:00,4910.0
42082,2026-05-08 20:15:00+00:00,4800.0
42083,2026-05-08 20:30:00+00:00,4710.0


In [23]:
res = process_raw_csv(input_path=path, output_path=output_path)
df_processed = res["df"]

In [24]:
df_processed


,load_forecast_mw
2022-12-31 22:00:00+00:00,4269.0
2022-12-31 23:00:00+00:00,4008.0
2023-01-01 00:00:00+00:00,3888.0
2023-01-01 01:00:00+00:00,3760.0
2023-01-01 02:00:00+00:00,3703.0
...,...
2026-05-08 16:00:00+00:00,5372.5
2026-05-08 17:00:00+00:00,5692.5
2026-05-08 18:00:00+00:00,5690.0
2026-05-08 19:00:00+00:00,5212.5


In [27]:
path = Path(r"C:\dev\greek-dam-forecasting\data\raw\backfill_tmp\renewable_forecast_historical.csv")
output_path = Path(r"C:\dev\greek-dam-forecasting\data\processed\renewable_forecast_historical_hourly.csv")

In [28]:
df_raw = pd.read_csv(path)
df_raw

,Unnamed: 0,solar,wind_onshore
0,2022-12-31 22:00:00+00:00,0.0,495.0
1,2022-12-31 23:00:00+00:00,0.0,518.0
2,2023-01-01 00:00:00+00:00,0.0,560.0
3,2023-01-01 01:00:00+00:00,0.0,603.0
4,2023-01-01 02:00:00+00:00,0.0,649.0
...,...,...,...
44597,2026-05-08 19:45:00+00:00,0.0,1770.0
44598,2026-05-08 20:00:00+00:00,0.0,1760.0
44599,2026-05-08 20:15:00+00:00,0.0,1740.0
44600,2026-05-08 20:30:00+00:00,0.0,1720.0


In [29]:
res = process_raw_csv(input_path=path, output_path=output_path)
df_processed = res["df"]
df_processed

,solar,wind_onshore
2022-12-31 22:00:00+00:00,0.0,495.0
2022-12-31 23:00:00+00:00,0.0,518.0
2023-01-01 00:00:00+00:00,0.0,560.0
2023-01-01 01:00:00+00:00,0.0,603.0
2023-01-01 02:00:00+00:00,0.0,649.0
...,...,...
2026-05-08 16:00:00+00:00,362.5,1755.0
2026-05-08 17:00:00+00:00,22.5,1810.0
2026-05-08 18:00:00+00:00,0.0,1827.5
2026-05-08 19:00:00+00:00,0.0,1792.5


In [31]:
import pandas as pd
path = Path(r"C:\dev\greek-dam-forecasting\data\raw\backfill_tmp\dam_prices_historical.csv")
df = pd.read_csv(path, index_col=0)
df.index = pd.to_datetime(df.index, utc=True)
print(f"Latest timestamp: {df.index.max()}")
print(f"Latest 5 rows:")
print(df.tail(5))

Latest timestamp: 2026-05-07 21:45:00+00:00
Latest 5 rows:
                           price_eur_mwh
2026-05-07 20:45:00+00:00         121.35
2026-05-07 21:00:00+00:00         113.84
2026-05-07 21:15:00+00:00         113.50
2026-05-07 21:30:00+00:00         113.42
2026-05-07 21:45:00+00:00         113.98


In [3]:
from src.data_loader import read_csv_from_blob

df = read_csv_from_blob("sagreekdamdevweu", "processed", "dam_prices/full.csv")
print(f"DAM full: {len(df)} rows, {df.index.min()} → {df.index.max()}")
print(df.head())
print(df.tail())

DAM full: 29328 rows, 2022-12-31 22:00:00+00:00 → 2026-05-06 21:00:00+00:00
                           price_eur_mwh
2022-12-31 22:00:00+00:00         210.05
2022-12-31 23:00:00+00:00         230.90
2023-01-01 00:00:00+00:00         268.19
2023-01-01 01:00:00+00:00         229.58
2023-01-01 02:00:00+00:00         235.98
                           price_eur_mwh
2026-05-06 17:00:00+00:00       179.5825
2026-05-06 18:00:00+00:00       151.0350
2026-05-06 19:00:00+00:00       159.5050
2026-05-06 20:00:00+00:00       161.7075
2026-05-06 21:00:00+00:00       167.4900


In [4]:
df

,price_eur_mwh
2022-12-31 22:00:00+00:00,210.0500
2022-12-31 23:00:00+00:00,230.9000
2023-01-01 00:00:00+00:00,268.1900
2023-01-01 01:00:00+00:00,229.5800
2023-01-01 02:00:00+00:00,235.9800
...,...
2026-05-06 17:00:00+00:00,179.5825
2026-05-06 18:00:00+00:00,151.0350
2026-05-06 19:00:00+00:00,159.5050
2026-05-06 20:00:00+00:00,161.7075


In [2]:
from src.dataset import load_combined_dataset

# Inner join (training): rows must have all three sources populated
prices_inner, exog_inner = load_combined_dataset(
    storage_account="sagreekdamdevweu",
    join="inner",
)

print(f"Inner: prices {len(prices_inner)}, exog {exog_inner.shape}")
print(f"  Range: {prices_inner.index.min()} → {prices_inner.index.max()}")
print(f"  Any NaN in prices: {prices_inner.isna().any()}")
print(f"  Any NaN in exog: {exog_inner.isna().any().any()}")

# Outer join (inference): includes timestamps where DAM is unknown
prices_outer, exog_outer = load_combined_dataset(
    storage_account="sagreekdamdevweu",
    join="outer",
)

print(f"\nOuter: prices {len(prices_outer)}, exog {exog_outer.shape}")
print(f"  Range: {prices_outer.index.min()} → {prices_outer.index.max()}")
print(f"  NaN prices count: {prices_outer.isna().sum()}")
print(f"  Any NaN in exog: {exog_outer.isna().any().any()}")

# Should see ~30k rows
prices_inner.head()

Inner: prices 29328, exog (29328, 3)
  Range: 2022-12-31 22:00:00+00:00 → 2026-05-06 21:00:00+00:00
  Any NaN in prices: False
  Any NaN in exog: True

Outer: prices 29375, exog (29375, 3)
  Range: 2022-12-31 22:00:00+00:00 → 2026-05-08 20:00:00+00:00
  NaN prices count: 47
  Any NaN in exog: True


2022-12-31 22:00:00+00:00    210.05
2022-12-31 23:00:00+00:00    230.90
2023-01-01 00:00:00+00:00    268.19
2023-01-01 01:00:00+00:00    229.58
2023-01-01 02:00:00+00:00    235.98
Freq: h, Name: price_eur_mwh, dtype: float64

In [3]:
prices_inner

2022-12-31 22:00:00+00:00    210.0500
2022-12-31 23:00:00+00:00    230.9000
2023-01-01 00:00:00+00:00    268.1900
2023-01-01 01:00:00+00:00    229.5800
2023-01-01 02:00:00+00:00    235.9800
                               ...   
2026-05-06 17:00:00+00:00    179.5825
2026-05-06 18:00:00+00:00    151.0350
2026-05-06 19:00:00+00:00    159.5050
2026-05-06 20:00:00+00:00    161.7075
2026-05-06 21:00:00+00:00    167.4900
Freq: h, Name: price_eur_mwh, Length: 29328, dtype: float64

In [4]:
exog_inner

,load_forecast_mw,solar,wind_onshore
2022-12-31 22:00:00+00:00,4269.0,0.0,495.0
2022-12-31 23:00:00+00:00,4008.0,0.0,518.0
2023-01-01 00:00:00+00:00,3888.0,0.0,560.0
2023-01-01 01:00:00+00:00,3760.0,0.0,603.0
2023-01-01 02:00:00+00:00,3703.0,0.0,649.0
...,...,...,...
2026-05-06 17:00:00+00:00,5725.0,30.0,732.5
2026-05-06 18:00:00+00:00,5872.5,0.0,747.5
2026-05-06 19:00:00+00:00,5382.5,0.0,730.0
2026-05-06 20:00:00+00:00,4922.5,0.0,702.5


In [5]:
from src.dataset import save_curated_dataset

url, blob_name = save_curated_dataset(
    storage_account="sagreekdamdevweu",
    join="inner",  # for training
)
print(f"Curated dataset uploaded: {url}")

Curated dataset uploaded: https://sagreekdamdevweu.blob.core.windows.net/curated/training_dataset.csv


In [7]:
prices_outer.tail(48)

2026-05-06 21:00:00+00:00    167.49
2026-05-06 22:00:00+00:00       NaN
2026-05-06 23:00:00+00:00       NaN
2026-05-07 00:00:00+00:00       NaN
2026-05-07 01:00:00+00:00       NaN
2026-05-07 02:00:00+00:00       NaN
2026-05-07 03:00:00+00:00       NaN
2026-05-07 04:00:00+00:00       NaN
2026-05-07 05:00:00+00:00       NaN
2026-05-07 06:00:00+00:00       NaN
2026-05-07 07:00:00+00:00       NaN
2026-05-07 08:00:00+00:00       NaN
2026-05-07 09:00:00+00:00       NaN
2026-05-07 10:00:00+00:00       NaN
2026-05-07 11:00:00+00:00       NaN
2026-05-07 12:00:00+00:00       NaN
2026-05-07 13:00:00+00:00       NaN
2026-05-07 14:00:00+00:00       NaN
2026-05-07 15:00:00+00:00       NaN
2026-05-07 16:00:00+00:00       NaN
2026-05-07 17:00:00+00:00       NaN
2026-05-07 18:00:00+00:00       NaN
2026-05-07 19:00:00+00:00       NaN
2026-05-07 20:00:00+00:00       NaN
2026-05-07 21:00:00+00:00       NaN
2026-05-07 22:00:00+00:00       NaN
2026-05-07 23:00:00+00:00       NaN
2026-05-08 00:00:00+00:00   

In [8]:
print("\nNaN counts per column (inner):")
print(exog_inner.isna().sum())
print("\nNaN counts per column (outer):")
print(exog_outer.isna().sum())

# When did the NaNs occur?
nan_rows = exog_inner[exog_inner.isna().any(axis=1)]
print(f"\n{len(nan_rows)} rows with at least one NaN in exog (inner)")
print(f"Sample:")
print(nan_rows.head())


NaN counts per column (inner):
load_forecast_mw    27
solar               27
wind_onshore        27
dtype: int64

NaN counts per column (outer):
load_forecast_mw    27
solar               27
wind_onshore        27
dtype: int64

27 rows with at least one NaN in exog (inner)
Sample:
                           load_forecast_mw  solar  wind_onshore
2023-12-31 22:00:00+00:00               NaN    NaN           NaN
2024-10-26 21:00:00+00:00               NaN    NaN           NaN
2024-10-26 22:00:00+00:00               NaN    NaN           NaN
2024-10-26 23:00:00+00:00               NaN    NaN           NaN
2024-10-27 00:00:00+00:00               NaN    NaN           NaN


In [15]:
exog_outer.index.diff().value_counts(dropna=False)

0 days 01:00:00    29374
NaT                    1
Name: count, dtype: int64